# 🏆 Notebook 06 — Model Evaluation & Selection

**Purpose**: Final evaluation on the **held-out test set**. Generate all visualisations. Write model cards.

**Theory**: Lecture 03 — All evaluation metrics. Lecture 01 — The model must generalise to unseen data.

**⚠️ IMPORTANT**: This is the ONLY notebook that touches the test set. Validation was for tuning. Test is for final honest evaluation.

In [ ]:
# Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib, json, os, sys, warnings
from sklearn.metrics import (mean_absolute_error, mean_squared_error, r2_score, max_error,
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    f1_score, precision_score, recall_score, roc_auc_score, RocCurveDisplay, PrecisionRecallDisplay)
from sklearn.preprocessing import StandardScaler
warnings.filterwarnings('ignore')
plt.rcParams['figure.dpi'] = 120
sns.set_style('whitegrid')
os.makedirs('../plots', exist_ok=True)
sys.path.insert(0, '..')
from src.feature_engineer import engineer_all_features
print('Ready.')

In [ ]:
# Load data and reproduce exact pipeline
df = pd.read_csv('../data/nev_battery_charging.csv').drop(columns=['timestamp']).drop_duplicates()
n = len(df); t_end, v_end = int(n*0.70), int(n*0.85)
targets = ['cycle_degradation','over_temp_flag','over_voltage_flag']
df_train = df.iloc[:t_end].copy(); df_val = df.iloc[t_end:v_end].copy(); df_test = df.iloc[v_end:].copy()
baseline_ir = df_train['internal_resistance'].iloc[0]

# Engineer features
df_train = engineer_all_features(df_train, baseline_ir, ['battery_temp'])
df_val = engineer_all_features(df_val, baseline_ir, ['battery_temp'])
df_test = engineer_all_features(df_test, baseline_ir, ['battery_temp'])

# Load selected features
with open('../models/feature_columns.json') as f: sel = json.load(f)
X_tr = df_train[[c for c in sel if c in df_train.columns]]
X_v = df_val[[c for c in sel if c in df_val.columns]]
X_te = df_test[[c for c in sel if c in df_test.columns]]

# Scale
scaler = StandardScaler()
scaler.fit(X_tr)
X_te_s = scaler.transform(X_te)
X_v_s = scaler.transform(X_v)

# Targets
y_reg_test = df_test['cycle_degradation']; y_reg_test_log = np.log1p(y_reg_test)
y_temp_test = df_test['over_temp_flag']; y_volt_test = df_test['over_voltage_flag']
print(f'Test set: {len(df_test)} rows, {X_te_s.shape[1]} features')

---
## 1. Regression — Final Test Evaluation

In [ ]:
# Load regression model
reg_model = joblib.load('../models/regression_model.pkl')
print(f'Loaded regression model: {type(reg_model).__name__}')

# Predict
y_pred_log = reg_model.predict(X_te_s)
y_pred = np.expm1(y_pred_log)
y_true = y_reg_test.values

# Metrics
mae = mean_absolute_error(y_true, y_pred)
rmse = np.sqrt(mean_squared_error(y_true, y_pred))
r2 = r2_score(y_true, y_pred)
max_err = max_error(y_true, y_pred)

print(f'\n=== REGRESSION TEST RESULTS ===')
print(f'  MAE:       {mae:.8f}')
print(f'  RMSE:      {rmse:.8f}')
print(f'  R²:        {r2:.4f}')
print(f'  Max Error: {max_err:.8f}')

In [ ]:
# Regression visualizations — 2x2 grid
fig, axes = plt.subplots(2, 2, figsize=(16, 14))

# 1. Actual vs Predicted scatter
axes[0,0].scatter(y_true, y_pred, alpha=0.5, s=20, c='#2196F3', edgecolors='white', linewidth=0.5)
lims = [min(y_true.min(), y_pred.min()), max(y_true.max(), y_pred.max())]
axes[0,0].plot(lims, lims, 'r--', linewidth=2, label='Perfect Prediction')
axes[0,0].set_xlabel('Actual cycle_degradation', fontsize=11)
axes[0,0].set_ylabel('Predicted cycle_degradation', fontsize=11)
axes[0,0].set_title('Actual vs Predicted', fontweight='bold', fontsize=13)
axes[0,0].legend()
axes[0,0].grid(True, alpha=0.3)

# 2. Residual plot
residuals = y_true - y_pred
axes[0,1].scatter(y_pred, residuals, alpha=0.5, s=20, c='#4CAF50', edgecolors='white', linewidth=0.5)
axes[0,1].axhline(y=0, color='red', linestyle='--', linewidth=2)
axes[0,1].set_xlabel('Predicted', fontsize=11)
axes[0,1].set_ylabel('Residual (Actual - Predicted)', fontsize=11)
axes[0,1].set_title('Residual Plot', fontweight='bold', fontsize=13)
axes[0,1].grid(True, alpha=0.3)

# 3. Residual distribution
axes[1,0].hist(residuals, bins=40, color='#FF9800', edgecolor='white', alpha=0.85)
axes[1,0].axvline(x=0, color='red', linestyle='--', linewidth=2)
axes[1,0].set_xlabel('Residual', fontsize=11)
axes[1,0].set_ylabel('Frequency', fontsize=11)
axes[1,0].set_title('Residual Distribution', fontweight='bold', fontsize=13)

# 4. Prediction over time
axes[1,1].plot(range(len(y_true)), y_true, 'b-', alpha=0.6, linewidth=1, label='Actual')
axes[1,1].plot(range(len(y_pred)), y_pred, 'r-', alpha=0.6, linewidth=1, label='Predicted')
axes[1,1].set_xlabel('Test Set Index', fontsize=11)
axes[1,1].set_ylabel('cycle_degradation', fontsize=11)
axes[1,1].set_title('Predictions Over Time', fontweight='bold', fontsize=13)
axes[1,1].legend()

plt.suptitle(f'Regression Evaluation — {type(reg_model).__name__}', fontweight='bold', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('../plots/eval_regression_results.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Feature importance for regression
try:
    if hasattr(reg_model, 'feature_importances_'):
        imp = reg_model.feature_importances_
        feat_names = [c for c in sel if c in df_test.columns]
        idx = np.argsort(imp)[::-1]
        fig, ax = plt.subplots(figsize=(12, 8))
        colors = plt.cm.RdYlGn(np.linspace(0.2, 0.8, len(idx)))
        ax.barh(range(len(idx)), imp[idx][::-1], color=colors)
        ax.set_yticks(range(len(idx)))
        ax.set_yticklabels([feat_names[i] for i in idx][::-1])
        ax.set_xlabel('Feature Importance', fontsize=12)
        ax.set_title('Best Regression Model — Feature Importances', fontweight='bold', fontsize=13)
        plt.tight_layout()
        plt.savefig('../plots/eval_regression_feature_importance.png', dpi=150, bbox_inches='tight')
        plt.show()
except Exception as e:
    print(f'Feature importance not available: {e}')

---
## 2. Classification — Final Test Evaluation

In [ ]:
# Load classification model and config
cls_model = joblib.load('../models/classification_model_temp.pkl')
with open('../models/classification_config.json') as f: config = json.load(f)
threshold = config['temp_threshold']
print(f'Loaded classifier: {type(cls_model).__name__}')
print(f'Threshold: {threshold}')
print(f'Voltage rule: {config["voltage_rule"]}')

# Predict
probs = cls_model.predict_proba(X_te_s)[:,1]
y_pred_cls = (probs >= threshold).astype(int)

# Metrics
print(f'\n=== CLASSIFICATION TEST RESULTS (over_temp_flag) ===')
print(f'  Precision: {precision_score(y_temp_test, y_pred_cls):.4f}')
print(f'  Recall:    {recall_score(y_temp_test, y_pred_cls):.4f}')
print(f'  F1:        {f1_score(y_temp_test, y_pred_cls):.4f}')
print(f'  ROC-AUC:   {roc_auc_score(y_temp_test, probs):.4f}')
print(f'\nClassification Report:')
print(classification_report(y_temp_test, y_pred_cls, target_names=['Normal','Over-Temp']))

In [ ]:
# Classification visualizations — 2x2 grid
fig, axes = plt.subplots(2, 2, figsize=(16, 14))

# 1. Confusion Matrix
ConfusionMatrixDisplay.from_predictions(y_temp_test, y_pred_cls,
    display_labels=['Normal','Over-Temp'], cmap='Blues', ax=axes[0,0])
axes[0,0].set_title(f'Confusion Matrix (threshold={threshold:.2f})', fontweight='bold')

# 2. ROC Curve
RocCurveDisplay.from_predictions(y_temp_test, probs, ax=axes[0,1], color='#2196F3', linewidth=2)
axes[0,1].plot([0,1],[0,1],'k--',alpha=0.5)
axes[0,1].set_title('ROC Curve (Test Set)', fontweight='bold')
axes[0,1].grid(True, alpha=0.3)

# 3. Precision-Recall Curve
PrecisionRecallDisplay.from_predictions(y_temp_test, probs, ax=axes[1,0], color='#4CAF50', linewidth=2)
from sklearn.metrics import precision_recall_curve
precs, recs, thrs = precision_recall_curve(y_temp_test, probs)
thr_idx = np.argmin(np.abs(thrs - threshold))
axes[1,0].plot(recs[thr_idx], precs[thr_idx], 'r*', markersize=20, label=f'Threshold={threshold:.2f}')
axes[1,0].legend()
axes[1,0].set_title('Precision-Recall Curve (Test Set)', fontweight='bold')
axes[1,0].grid(True, alpha=0.3)

# 4. Probability distribution
axes[1,1].hist(probs[y_temp_test==0], bins=30, alpha=0.6, color='#4CAF50', label='Normal', edgecolor='white')
axes[1,1].hist(probs[y_temp_test==1], bins=30, alpha=0.6, color='#F44336', label='Over-Temp', edgecolor='white')
axes[1,1].axvline(x=threshold, color='black', linestyle='--', linewidth=2, label=f'Threshold={threshold:.2f}')
axes[1,1].set_xlabel('Predicted Probability', fontsize=11)
axes[1,1].set_ylabel('Count', fontsize=11)
axes[1,1].set_title('Probability Distribution by Class', fontweight='bold')
axes[1,1].legend()

plt.suptitle('Classification Evaluation — over_temp_flag', fontweight='bold', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('../plots/eval_classification_results.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 3. Model Comparison Summary Tables

In [ ]:
# Load all results
try:
    with open('../models/regression_results.json') as f: reg_results = json.load(f)
except: reg_results = {}
try:
    with open('../models/classification_results.json') as f: cls_results = json.load(f)
except: cls_results = {}

print('=== REGRESSION MODEL COMPARISON (Validation) ===')
if reg_results:
    reg_df = pd.DataFrame(reg_results).T
    print(reg_df.to_string())

print(f'\n=== FINAL TEST RESULTS ===')
print(f'Regression ({type(reg_model).__name__}):')
print(f'  MAE={mae:.8f}, RMSE={rmse:.8f}, R²={r2:.4f}, MaxError={max_err:.8f}')
print(f'\nClassification ({type(cls_model).__name__}, threshold={threshold}):')
print(f'  Recall={recall_score(y_temp_test,y_pred_cls):.4f}, Precision={precision_score(y_temp_test,y_pred_cls):.4f}')
print(f'  F1={f1_score(y_temp_test,y_pred_cls):.4f}, ROC-AUC={roc_auc_score(y_temp_test,probs):.4f}')

In [ ]:
# Final summary visualization
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Regression metrics
reg_metrics = {'MAE': mae, 'RMSE': rmse, 'R²': r2}
colors_r = ['#2196F3', '#4CAF50', '#FF9800']
axes[0].bar(reg_metrics.keys(), reg_metrics.values(), color=colors_r, edgecolor='white', linewidth=2)
for i, (k,v) in enumerate(reg_metrics.items()):
    axes[0].text(i, v + v*0.02, f'{v:.6f}', ha='center', fontsize=10)
axes[0].set_title('Regression Test Metrics', fontweight='bold', fontsize=13)

# Classification metrics
cls_metrics = {'Recall': recall_score(y_temp_test,y_pred_cls),
               'Precision': precision_score(y_temp_test,y_pred_cls),
               'F1': f1_score(y_temp_test,y_pred_cls),
               'ROC-AUC': roc_auc_score(y_temp_test,probs)}
colors_c = ['#F44336', '#9C27B0', '#00BCD4', '#FF9800']
axes[1].bar(cls_metrics.keys(), cls_metrics.values(), color=colors_c, edgecolor='white', linewidth=2)
for i, (k,v) in enumerate(cls_metrics.items()):
    axes[1].text(i, v + 0.02, f'{v:.4f}', ha='center', fontsize=10)
axes[1].set_ylim(0, 1.15)
axes[1].set_title('Classification Test Metrics', fontweight='bold', fontsize=13)

plt.suptitle('Final Model Performance — Test Set', fontweight='bold', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('../plots/eval_final_summary.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 4. Model Cards

In [ ]:
# Generate Model Cards
reg_card = f'''# Model Card — Regression Model

## Model: {type(reg_model).__name__}
**Task**: Predict cycle_degradation (battery degradation per charge cycle)

## Training Data
- Source: nev_battery_charging.csv
- Training rows: 0 to {t_end-1} ({t_end} samples)
- Chronological split (no shuffle)

## Features
- {len(sel)} selected features after RFE
- Features: {sel}

## Target Transform
- log1p applied during training
- expm1 applied to predictions for original-scale metrics

## Test Set Performance
- MAE: {mae:.8f}
- RMSE: {rmse:.8f}
- R²: {r2:.4f}
- Max Error: {max_err:.8f}

## Known Limitations
- Trained on 1,330 samples — limited dataset size
- Performance may degrade on battery chemistries not in training data
- cycle_degradation values are extremely small (1e-4 to 1e-3)
- Model assumes time-ordered data from same battery lifecycle
'''

cls_card = f'''# Model Card — Classification Model (over_temp_flag)

## Model: {type(cls_model).__name__}
**Task**: Predict over-temperature events (binary: 0=Normal, 1=Over-Temp)

## Training Data
- Source: nev_battery_charging.csv
- Training rows: 0 to {t_end-1}
- Class imbalance handled via class_weight="balanced" / scale_pos_weight

## Decision Threshold
- Optimized threshold: {threshold}
- Tuned for Recall >= 0.85 (minimizing missed over-temp events)

## Test Set Performance
- Recall: {recall_score(y_temp_test, y_pred_cls):.4f}
- Precision: {precision_score(y_temp_test, y_pred_cls):.4f}
- F1: {f1_score(y_temp_test, y_pred_cls):.4f}
- ROC-AUC: {roc_auc_score(y_temp_test, probs):.4f}

## over_voltage_flag
- Rule-based fallback: flag=1 if action_voltage > 4.15 OR terminal_voltage > 4.18
- Reason: insufficient positive training samples (<20)

## Known Limitations
- Temporal block pattern in labels may not generalize to random over-temp events
- class imbalance affects precision
- Rule-based voltage fallback lacks learned nuance
'''

print(reg_card)
print('='*60)
print(cls_card)

# Save model cards
with open('../models/MODEL_CARD.md', 'w') as f:
    f.write(reg_card + '\n---\n\n' + cls_card)
print('\nSaved: models/MODEL_CARD.md')

## ✅ Evaluation Complete

All models have been evaluated on the held-out test set with full visualizations saved to `plots/`.

### Key Files Generated:
- `plots/eval_regression_results.png` — Actual vs Predicted, Residuals, Distribution
- `plots/eval_classification_results.png` — Confusion Matrix, ROC, PR Curve
- `plots/eval_final_summary.png` — Combined metrics overview
- `models/MODEL_CARD.md` — Professional model documentation